In [1]:
import os,sys,time
 

os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYSPARK_PYTHON"]        = sys.executable


In [2]:
from pyspark import SparkContext
from sympy import *
from drudge import *
#from gristmill import *
print("finished importing")


finished importing


In [3]:
import re

# ----------------------------------------------------------------------------
# 1) Start Spark & BCS quasiparticle drudge
# ----------------------------------------------------------------------------
ctx = SparkContext('local[*]', 'bcs_ccsd')
dr  = ReducedBCSDrudge(ctx)
dr.full_simplify = True
print("finished")

26/08/26 21:44:07 WARN Utils: Your hostname, Swarnamoys-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.0.0.87 instead (on interface en0)
26/08/26 21:44:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/26 21:44:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

finished


In [4]:
u, v = IndexedBase('u'), IndexedBase('v')
x = Symbol('x')
H00 = Symbol('H00')
P, Pdag, N = dr.names.P, dr.names.Pdag, dr.names.N
p,q,r,s,i,j,k,l = dr.names.A_dumms[:8]
P, Pdag, N = dr.names.P, dr.names.Pdag, dr.names.N
h = IndexedBase('h')
Y = IndexedBase('Y')
W = IndexedBase('W')
V = IndexedBase('V')
dr.set_symm(W, Perm([1, 0]), valence=2)
#zero_term_w = [(W[p,p],0)]

In [5]:
 
# -------------------------
# P^\dagger
# -------------------------
P_i_dag = (u[p]*v[p]+ u[p]**2 * Pdag[p]- u[p]*v[p] * N[p]- v[p]**2 * P[p])

P_j_dag = (u[q]*v[q]+ u[q]**2 * Pdag[q]- u[q]*v[q] * N[q]- v[q]**2 * P[q])

P_k_dag = (u[r]*v[r]+ u[r]**2 * Pdag[r]- u[r]*v[r] * N[r]- v[r]**2 * P[r])

# -------------------------
# P
# P_j = u_j^* v_j^* + (u_j^*)^2 P_j - u_j^* v_j^* N_j - (v_j^*)^2 P_j^\dagger
# -------------------------
P_i = (conjugate(u[p]) * conjugate(v[p])+ conjugate(u[p])**2 * P[p]- conjugate(u[p]) * conjugate(v[p]) * N[p]
    - conjugate(v[p])**2 * Pdag[p]
)

P_j = (
      conjugate(u[q]) * conjugate(v[q])
    + conjugate(u[q])**2 * P[q]
    - conjugate(u[q]) * conjugate(v[q]) * N[q]
    - conjugate(v[q])**2 * Pdag[q]
)

P_k = (
      conjugate(u[r]) * conjugate(v[r])
    + conjugate(u[r])**2 * P[r]
    - conjugate(u[r]) * conjugate(v[r]) * N[r]
    - conjugate(v[r])**2 * Pdag[r]
)

# -------------------------
# N
# N_i = 2 v_i^* v_i + (u_i^* u_i - v_i^* v_i) N_i
#       + 2 u_i v_i^* P_i^\dagger + 2 u_i^* v_i P_i
# -------------------------
N_i = (
      2 * conjugate(v[p]) * v[p]
    + (conjugate(u[p]) * u[p] - conjugate(v[p]) * v[p]) * N[p]
    + 2 * u[p] * conjugate(v[p]) * Pdag[p]
    + 2 * conjugate(u[p]) * v[p] * P[p]
)

N_j = (
      2 * conjugate(v[q]) * v[q]
    + (conjugate(u[q]) * u[q] - conjugate(v[q]) * v[q]) * N[q]
    + 2 * u[q] * conjugate(v[q]) * Pdag[q]
    + 2 * conjugate(u[q]) * v[q] * P[q]
)

N_k = (
      2 * conjugate(v[r]) * v[r]
    + (conjugate(u[r]) * u[r] - conjugate(v[r]) * v[r]) * N[r]
    + 2 * u[r] * conjugate(v[r]) * Pdag[r]
    + 2 * conjugate(u[r]) * v[r] * P[r]
)

In [ ]:
X = dr.sum(N_i)
X.display()

In [ ]:
H11, H20 , H02 = IndexedBase('H11'), IndexedBase('H20'), IndexedBase('H02')
H04, H40 = IndexedBase('H04'), IndexedBase('H40')
H22, Hb22 = IndexedBase('H22'), IndexedBase('HT22')
H31, H13 = IndexedBase('H31'), IndexedBase('H13')
 
dr.set_symm(H04,Perm([1,0],IDENT))
dr.set_symm(H40,Perm([1,0],IDENT)) 

In [ ]:
#expr = H11[p]*N[p] + H02[p]*Pdag[p] +H20[p]*P[p]+H22[p,q]*N[p]*N[q]+Hb22[p,q]*Pdag[p]*P[q]\
#    +H40[p,q]*P[p]*P[q]+H04[p,q]*Pdag[p]*Pdag[q]+H13[p,q]*Pdag[p]*N[q]+H31[p,q]*N[p]*P[q]
#H_N = dr.einst(expr).simplify().merge()

In [8]:
expr = h[p]*N_i + V[p,q]*P_i_dag*P_j+ W[p,q]*N_i*N_j
expr = dr.einst(expr).simplify().merge()
#expr = (expr.subst_all(zero_term_w)).merge()
expr.display()


<IPython.core.display.Math object>

In [ ]:
print(expr.latex())

In [9]:
def coeff_with_internal_sums(ts):
    """
    Strip the operator word and *only* those sums that run over
    indices appearing in the operator word. Keep any other internal sums. We need this for Integrals.
    """
    def proc(t):
        # Collect indices used in the operator word
        op_inds = set()
        for v in t.vecs:
            # depending on your Drudge version, this may be v.indices or v.inds
            for ind in v.indices:
                op_inds.add(ind)

        # Keep only sums whose index is NOT in the operator word
        new_sums = tuple((i, base) for (i, base) in t.sums if i not in op_inds)

        # Strip vecs; keep amp unchanged
        return Term(new_sums, t.amp, ())

    return ts.map(proc)

def coeff_only(ts):
    """Return a TermSum with the same sums/amp but with vecs removed."""
    # ts can be a TermSum or anything with .map over terms
    return ts.map(lambda t: Term(t.sums, t.amp, ()))   # vecs = ()

In [10]:
def get_PdagPdagP(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
    # Normal-ordered triple: P† P† P
    return labels == ['P^\\dagger', 'P^\\dagger','P']

def get_PdagPP(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger', 'P','P']

def get_NNN(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['N', 'N','N']


def get_PPP(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P', 'P','P']


def get_PdagPdagN(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger', 'P^\\dagger','N']


def get_NNP(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['N', 'N','P']


def get_NPP(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['N', 'P','P']


def get_PdagPdagPdag(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger','P^\\dagger','P^\\dagger']


def get_PdagNN(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger', 'N','N']


def get_PdagNP(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger', 'N','P']

def get_Pdag(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 1:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger']

def get_P(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 1:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P']

def get_N(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 1:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['N']

def get_NN(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['N','N']

def get_NP(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['N','P']

def get_PP(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P','P']

def get_PdagPdag(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger','P^\\dagger']

def get_PdagN(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger','N']

def get_PdagP(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger','P']



In [ ]:
def drop_terms_containing_P(expr):

    ''' Assuming expr is normal_ordered '''
    kept = expr.simplify().filter(lambda term: all(v.label != 'P' for v in term.vecs))
     
    #return Tensor(dr,kept).simplify().merge()
    return kept

def keep_exactly_k_Pdag(expr, k ,pdag_label='P^\\dagger'):

    """
    Keep only terms with exactly k creators P^\\dagger in the operator string.
    Assumes expr has already had all P (annihilators) filtered out and is normal ordered.
    """
    def nPdag(term):
        return sum(1 for v in term.vecs if v.label == pdag_label)

    kept = expr.simplify().filter(lambda term: nPdag(term) == k)
    #return Tensor(dr, kept).simplify().merge()
    return kept
    

In [14]:
tensor0_ = expr.filter(lambda term: len(term.vecs) == 0)
#tensor0_ = dr.einst(tensor0_).simplify().merge().merge()
tensor0_ = tensor0_.simplify().merge().merge()
 
tensor0_.display()

<IPython.core.display.Math object>

In [15]:
tensor1_ = expr.filter(lambda term: len(term.vecs) == 1)

#tensor1_ = dr.einst(tensor1_).simplify().merge().merge()
tensor1_ = tensor1_.simplify().merge().merge()
tensor1_.display()


<IPython.core.display.Math object>

In [17]:
Np_term = tensor1_.terms.filter(get_N)
H010_dr = Tensor(dr,  coeff_with_internal_sums(Np_term)).simplify().merge()
H010_dr.display()

<IPython.core.display.Math object>

In [18]:
print(H010_dr.latex())

- \sum_{q \in A} \left(\overline{u_{p}} \overline{v_{p}} V_{q,p} u_{q} v_{q} - 4 \overline{u_{p}} \overline{v_{q}} W_{q,p} u_{p} v_{q} + \overline{u_{q}} \overline{v_{q}} V_{p,q} u_{p} v_{p} + 4 \overline{v_{p}} \overline{v_{q}} W_{q,p} v_{p} v_{q}\right)  - \left(4 \overline{u_{p}} \overline{v_{p}} W_{p,p} u_{p} v_{p} - \overline{u_{p}} h_{p} u_{p} + \overline{v_{p}}^{2} V_{p,p} v_{p}^{2} + \overline{v_{p}} h_{p} v_{p}\right) 


In [19]:
Pp_term = tensor1_.terms.filter(get_P)
H001_dr = Tensor(dr,  coeff_with_internal_sums(Pp_term)).simplify().merge()
H001_dr.display()

<IPython.core.display.Math object>

In [20]:
print(H001_dr.latex())

\sum_{q \in A} \left(\overline{u_{p}}^{2} V_{q,p} u_{q} v_{q} + 8 \overline{u_{p}} \overline{v_{q}} W_{q,p} v_{p} v_{q} - \overline{u_{q}} \overline{v_{q}} V_{p,q} v_{p}^{2}\right)   + \left(4 \overline{u_{p}}^{2} W_{p,p} u_{p} v_{p} + 2 \overline{u_{p}} \overline{v_{p}} V_{p,p} v_{p}^{2} - 4 \overline{u_{p}} \overline{v_{p}} W_{p,p} v_{p}^{2} + 2 \overline{u_{p}} h_{p} v_{p}\right) 


In [21]:
Pdagp_term = tensor1_.terms.filter(get_Pdag)
H100_dr = Tensor(dr,  coeff_with_internal_sums(Pdagp_term)).simplify().merge()
H100_dr.display()

<IPython.core.display.Math object>

In [22]:
print(H100_dr.latex())

\sum_{q \in A} \left(\overline{u_{q}} \overline{v_{q}} V_{p,q} u_{p}^{2} - \overline{v_{p}}^{2} V_{q,p} u_{q} v_{q} + 8 \overline{v_{p}} \overline{v_{q}} W_{q,p} u_{p} v_{q}\right)   + \left(4 \overline{u_{p}} \overline{v_{p}} W_{p,p} u_{p}^{2} + 2 \overline{v_{p}}^{2} V_{p,p} u_{p} v_{p} - 4 \overline{v_{p}}^{2} W_{p,p} u_{p} v_{p} + 2 \overline{v_{p}} h_{p} u_{p}\right) 


In [23]:
tensor2_ = expr.filter(lambda term: len(term.vecs) == 2)

#tensor1_ = dr.einst(tensor1_).simplify().merge().merge()
tensor2_ = tensor2_.simplify().merge().merge()
tensor2_.display()


<IPython.core.display.Math object>

In [25]:
PdagP_term = tensor2_.terms.filter(get_PdagP)
H101_dr = Tensor(dr,  coeff_with_internal_sums(PdagP_term)).simplify().merge()
H101_dr.display()

<IPython.core.display.Math object>

In [26]:
print(H101_dr.latex())

\left(\overline{u_{q}}^{2} V_{p,q} u_{p}^{2} + 8 \overline{u_{q}} \overline{v_{p}} W_{p,q} u_{p} v_{q} + \overline{v_{p}}^{2} V_{q,p} v_{q}^{2}\right) 


In [27]:
PdagPdag_term = tensor2_.terms.filter(get_PdagPdag)
H200_dr = Tensor(dr,  coeff_with_internal_sums(PdagPdag_term)).simplify().merge()
H200_dr.display()

<IPython.core.display.Math object>

In [28]:
PP_term = tensor2_.terms.filter(get_PP)
H002_dr = Tensor(dr,  coeff_with_internal_sums(PP_term)).simplify().merge()
H002_dr.display()

<IPython.core.display.Math object>

In [29]:
PdagN_term = tensor2_.terms.filter(get_PdagN)
H110_dr = Tensor(dr,  coeff_with_internal_sums(PdagN_term)).simplify().merge()
H110_dr.display()

<IPython.core.display.Math object>

In [30]:
print(H110_dr.latex())

\left(4 \overline{u_{q}} \overline{v_{p}} W_{p,q} u_{p} u_{q} - \overline{u_{q}} \overline{v_{q}} V_{p,q} u_{p}^{2} + \overline{v_{p}}^{2} V_{q,p} u_{q} v_{q} - 4 \overline{v_{p}} \overline{v_{q}} W_{p,q} u_{p} v_{q}\right) 


In [31]:
NP_term = tensor2_.terms.filter(get_NP)
H011_dr = Tensor(dr,  coeff_with_internal_sums(NP_term)).simplify().merge()
H011_dr.display()

<IPython.core.display.Math object>

In [32]:
print(H011_dr.latex())

\left(4 \overline{u_{p}} \overline{u_{q}} W_{p,q} u_{p} v_{q} + \overline{u_{p}} \overline{v_{p}} V_{q,p} v_{q}^{2} - \overline{u_{q}}^{2} V_{p,q} u_{p} v_{p} - 4 \overline{u_{q}} \overline{v_{p}} W_{p,q} v_{p} v_{q}\right) 


In [33]:
NN_term = tensor2_.terms.filter(get_NN)
H020_dr = Tensor(dr,  coeff_with_internal_sums(NN_term)).simplify().merge()
H020_dr.display()

<IPython.core.display.Math object>

In [34]:
print(H020_dr.latex())

\left(\overline{u_{p}} \overline{u_{q}} W_{p,q} u_{p} u_{q} - 2 \overline{u_{p}} \overline{v_{q}} W_{p,q} u_{p} v_{q} + \overline{u_{q}} \overline{v_{q}} V_{p,q} u_{p} v_{p} + \overline{v_{p}} \overline{v_{q}} W_{p,q} v_{p} v_{q}\right) 
